In [1]:
import os
import gc
import sys
import matplotlib.pyplot as plt
import threading
import time
import numpy as np

 #from scipy.ndimage import gaussian_filter
if sys.platform!='darwin':
    from cupyx.scipy.ndimage import gaussian_filter
    import cupy as cp
else:
    from scipy.ndimage import gaussian_filter
    import pyclesperanto_prototype as cle
    # initialize GPU
    device = cle.select_device("GTX")
    print("Used GPU: ", device)
 

import numpy as np
from os.path import getsize
import napari
from skimage.io import imread

filename =  'Image_001_001.raw'
previewFilename = 'ChanC_Preview.tif'
MAXCHUNKSIZE = 1024*288*2*3

class thorlabsFile():
   
    def __init__(self,folder) -> None:

        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)

        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)
        self.app = napari.Viewer()
        

        #self.app.add_image(self.array)

    def loadFile(self,folder):

        try:
            self.r.close()
        except:
            pass
        
        self.folder = folder
        self.fullpath = os.path.join(self.folder,filename)
        prev = imread(os.path.join(self.folder,previewFilename))

        self.width = prev.shape[1] 
        self.height = prev.shape[0]

        self.r = open(self.fullpath,'rb')
        nbytes = getsize(self.fullpath)
        self.frameSize = self.width*self.height*2
        self.nFrames = int(nbytes/self.frameSize)

        self.currentLastFrame = 0

        self.array = np.empty((0,self.height,self.width),dtype=np.uint16)

        l = self.app.layers['Image']
        
        l.data = self.array
        
    def getImage(self,n):

        offset = n*self.frameSize
        self.r.seek(offset)
        st = self.r.read(self.frameSize)
        nparray = np.frombuffer(st,dtype = np.uint16).reshape((1,self.height,self.width))
        
        return nparray

    def loadWholeStack(self,start=0,end=-1,step=1):

        
        if end == -1:
            end = self.nFrames  # Fixed: was 'nFrames' should be 'self.nFrames'
        
        # Add validation to prevent crashes
        if start >= end or start < 0 or end > self.nFrames:
            print(f"Invalid frame range: start={start}, end={end}, nFrames={self.nFrames}")
            return []
            
        totalFrames = end-start
        totalFramesSize = totalFrames*self.frameSize

        if totalFramesSize<=MAXCHUNKSIZE:
            #stack = np.zeros((self.height,self.width,totalSize),dtype=np.uint16)
            #for i in range(start,end,step):
            #    stack[:,:,i-start] = gaussian_filter(self.getImage(i),2)
            #    #stack.append(getImage(r,i,width,height))

            offset = start*self.frameSize
            self.r.seek(offset)
            st = self.r.read(totalFramesSize)
            
            if sys.platform != 'darwin':
                stack = cp.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))
                stack = stack[::,:,:]
                stack = gaussian_filter(stack,(2,2,2))
                stack3 = [cp.asnumpy(stack)]
            else:
                stack = np.frombuffer(st,dtype = np.uint16).reshape((totalFrames,self.height,self.width))
                stack = stack[::2,:,:]
                stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                stack3 = [stack]


            
        else:
            chunksizeFrames = MAXCHUNKSIZE//(self.frameSize)  #number of frames in a chunk
            nchunks = totalFrames//chunksizeFrames
            remainderFrames = totalFrames%chunksizeFrames
            stack3 = []
            for i in range(nchunks):
                offset = (start+i*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*chunksizeFrames)
                if sys.platform !='darwin':
                    stack = cp.frombuffer(st,dtype = np.uint16).reshape((chunksizeFrames,self.height,self.width))
                    stack = stack[::,:,:]
                    stack = gaussian_filter(stack,(2,2,2))
                    stack3.append(cp.asnumpy(stack))
                else:
                    stack = np.frombuffer(st,dtype = np.uint16).reshape((chunksizeFrames,self.height,self.width))
                    stack = stack[::2,:,:]
                    stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                    stack3.append(stack)

            if remainderFrames != 0:
                offset = (start+nchunks*chunksizeFrames)*self.frameSize
                self.r.seek(offset)
                st = self.r.read(self.frameSize*remainderFrames)
                if sys.platform != 'darwin':
                    stack = cp.frombuffer(st,dtype = np.uint16).reshape((remainderFrames,self.height,self.width))
                    stack = stack[::,:,:]
                    stack = gaussian_filter(stack,(2,2,2))
                    stack3.append(cp.asnumpy(stack))
                else:
                    stack = np.frombuffer(st,dtype = np.uint16).reshape((remainderFrames,self.height,self.width))
                    stack = stack[::,:,:]
                    stack = cle.gaussian_blur(stack,sigma_x=2,sigma_y=2,sigma_z=2).astype(np.uint16)
                    stack3.append(stack) 
        if sys.platform !='darwin':         
            cp._default_memory_pool.free_all_blocks()    
        return stack3

    def loadNextNFrames(self,n):
        newStacks = self.loadWholeStack(start=self.currentLastFrame,end = self.currentLastFrame+n)
        self.array= np.vstack([self.array, *newStacks])
        del newStacks
        try:
            l = self.app.layers['Image']
            l.data = self.array
        except:
            self.app.add_image(self.array)
            self.app.add_shapes(None, shape_type='rectangle', name='Shapes',  edge_width=3, face_color=np.array([0,0,0,0]),edge_color = 'red',edge_color_cycle=plt.rcParams['axes.prop_cycle'].by_key()['color'])


        self.currentLastFrame = self.array.shape[0]

    def loadUpToFrameN(self,n):
        
        if (n<self.currentLastFrame) | (n>self.nFrames):
            return []  # Return empty list instead of None
        newStack = self.loadWholeStack(start=self.currentLastFrame,end = n)
        self.array= np.vstack([self.array, *newStack])

        if self.currentLastFrame == 0:
                self.app.add_image(self.array)
                self.app.add_shapes(None, shape_type='rectangle', name='Shapes',  edge_width=3, face_color=np.array([0,0,0,0]),edge_color = 'red',edge_color_cycle=plt.rcParams['axes.prop_cycle'].by_key()['color'])
        else:
            l = self.app.layers['Image']
            l.data = self.array
        self.currentLastFrame = self.array.shape[0]

    def start_live_monitoring(self, chunk_size=10, wait_time=1.0, fps=15):
        """
        Monitor a file being written in real-time (e.g., 15 fps acquisition).
        This is a SYNCHRONOUS, crash-proof version that manually checks for new frames.
        Call repeatedly to continue monitoring.
        
        Args:
            chunk_size: Number of frames to load at once
            wait_time: Time to wait when hitting black frames (seconds)
            fps: Expected acquisition frame rate (for info only)
        
        Returns:
            dict: Status information about the monitoring attempt
        """
        # Check if we've reached the end of allocated file
        if self.currentLastFrame >= self.nFrames:
            return {"status": "finished", "message": "Reached end of allocated file space", "frames_loaded": 0}
        
        # Load next chunk
        end_frame = min(self.currentLastFrame + chunk_size, self.nFrames)
        print(f"Checking frames {self.currentLastFrame} to {end_frame-1}")
        
        try:
            newStacks = self.loadWholeStack(start=self.currentLastFrame, end=end_frame)
            if not newStacks:
                return {"status": "error", "message": "Failed to load stack", "frames_loaded": 0}
            
            new_block = np.vstack(newStacks)
            
            # Check for black frames (not yet acquired)
            zero_idx = None
            for i, frame in enumerate(new_block):
                if np.all(frame == 0):
                    zero_idx = i
                    break
            
            if zero_idx is not None:
                if zero_idx > 0:
                    # Some frames have data, some are black
                    new_block = new_block[:zero_idx]
                    frames_added = new_block.shape[0]
                    
                    # Add frames and update viewer
                    self.array = np.vstack([self.array, new_block])
                    self._safe_update_viewer()
                    self.currentLastFrame = self.array.shape[0]
                    
                    return {
                        "status": "partial_live_edge", 
                        "message": f"Loaded {frames_added} frames, hit live edge", 
                        "frames_loaded": frames_added,
                        "total_frames": self.currentLastFrame
                    }
                else:
                    # All frames are black - we're at the live edge
                    adaptive_wait = min(wait_time, (chunk_size / fps) * 1.5) if fps > 0 else wait_time
                    return {
                        "status": "live_edge", 
                        "message": f"At live edge - wait {adaptive_wait:.1f}s before calling again", 
                        "frames_loaded": 0,
                        "wait_time": adaptive_wait
                    }
            else:
                # All frames have data
                frames_added = new_block.shape[0]
                self.array = np.vstack([self.array, new_block])
                self._safe_update_viewer()
                self.currentLastFrame = self.array.shape[0]
                
                return {
                    "status": "success", 
                    "message": f"Loaded {frames_added} frames successfully", 
                    "frames_loaded": frames_added,
                    "total_frames": self.currentLastFrame
                }
                
        except Exception as e:
            return {"status": "error", "message": f"Error loading frames: {e}", "frames_loaded": 0}
    
    def _safe_update_viewer(self):
        """Safely update the viewer without any threading issues"""
        try:
            if hasattr(self.app, 'layers') and len(self.app.layers) > 0:
                # Find existing image layer
                for layer in self.app.layers:
                    if hasattr(layer, 'data'):
                        layer.data = self.array
                        break
            else:
                # Create new layer
                self.app.add_image(self.array, name='Live Stream')
                try:
                    self.app.add_shapes(
                        None, shape_type='rectangle', name='Shapes',
                        edge_width=3, face_color=np.array([0,0,0,0]),
                        edge_color='red',
                        edge_color_cycle=plt.rcParams['axes.prop_cycle'].by_key()['color']
                    )
                except:
                    pass  # Shapes might already exist
            print(f"✓ Viewer updated - showing {self.currentLastFrame} frames")
        except Exception as e:
            print(f"Viewer update warning: {e}")
    
    def auto_monitor_loop(self, chunk_size=10, wait_time=1.0, fps=15, max_iterations=100):
        """
        Convenience method to automatically monitor in a loop.
        Still synchronous but handles the waiting logic.
        """
        print(f"Starting auto-monitoring loop (max {max_iterations} iterations)")
        print(f"Chunk size: {chunk_size}, wait time: {wait_time}s, expected fps: {fps}")
        
        for iteration in range(max_iterations):
            result = self.start_live_monitoring(chunk_size=chunk_size, wait_time=wait_time, fps=fps)
            
            print(f"Iteration {iteration+1}: {result['message']}")
            
            if result['status'] == 'finished':
                print("Monitoring completed - reached end of file")
                break
            elif result['status'] == 'error':
                print(f"Monitoring stopped due to error: {result['message']}")
                break
            elif result['status'] == 'live_edge':
                # Wait before next attempt
                wait_time_actual = result.get('wait_time', wait_time)
                print(f"Waiting {wait_time_actual:.1f}s at live edge...")
                time.sleep(wait_time_actual)
            elif result['status'] in ['success', 'partial_live_edge']:
                # Continue immediately for next chunk
                time.sleep(0.1)  # Small pause to prevent overload
        
        print(f"Auto-monitoring loop completed after {iteration+1} iterations")
        return self.currentLastFrame
    
    def update_viewer(self):
        """Force refresh the viewer display"""
        try:
            if hasattr(self.app, 'layers') and len(self.app.layers) > 0:
                for layer in self.app.layers:
                    if hasattr(layer, 'data'):
                        layer.data = self.array
                        break
            print(f"Viewer refreshed - showing {self.currentLastFrame} frames")
            return True
        except Exception as e:
            print(f"Viewer refresh error: {e}")
            return False
    
    def update_from_background(self):
        """Legacy method - use update_viewer() instead"""
        return self.update_viewer()
            
    def get_status(self):
        """Get current status"""
        print(f"Current status: {self.currentLastFrame} frames loaded out of {self.nFrames} total")
        if self.currentLastFrame < self.nFrames:
            print(f"Ready to load more frames - call tb.start_live_monitoring() to continue")
        else:
            print("All frames loaded")
        return self.currentLastFrame

Used GPU:  <Apple M1 Pro on Platform: Apple (2 refs)>


/Users/federico/miniforge3/envs/napari/lib/python3.13/site-packages/pyclesperanto_prototype/_tier0/_device.py:77: UserWarning: No OpenCL device found with GTX in their name. Using Apple M1 Pro instead.
  warnings.warn(f"No OpenCL device found with {name} in their name. Using {device.name} instead.")


In [2]:
tb = thorlabsFile('../sampleImage')

In [3]:
# Test the NEW crash-proof live monitoring feature
# This version is completely synchronous and won't crash the kernel

# Option 1: Single monitoring check (call repeatedly)
result = tb.start_live_monitoring(chunk_size=5, wait_time=0.5, fps=15)
print("\nResult:", result)

Checking frames 0 to 4
✓ Viewer updated - showing 0 frames

Result: {'status': 'success', 'message': 'Loaded 4 frames successfully', 'frames_loaded': 4, 'total_frames': 4}
✓ Viewer updated - showing 0 frames

Result: {'status': 'success', 'message': 'Loaded 4 frames successfully', 'frames_loaded': 4, 'total_frames': 4}


In [4]:
# Check the current status of live monitoring
tb.get_status()

Current status: 4 frames loaded out of 1000 total
Ready to load more frames - call tb.start_live_monitoring() to continue


4

In [3]:
# Option 2: Auto-monitoring loop (completely safe, no threading)
# This will automatically check for new frames and wait at live edges
# Stops after max_iterations or when reaching end of file

tb.auto_monitor_loop(chunk_size=50, wait_time=10, fps=15, max_iterations=10000)

Starting auto-monitoring loop (max 10000 iterations)
Chunk size: 50, wait time: 10s, expected fps: 15
Checking frames 0 to 49
✓ Viewer updated - showing 0 frames
Iteration 1: Loaded 8 frames, hit live edge
Checking frames 8 to 57
✓ Viewer updated - showing 8 frames
Iteration 2: Loaded 4 frames, hit live edge
Checking frames 12 to 61
✓ Viewer updated - showing 12 frames
Iteration 3: Loaded 2 frames, hit live edge
Checking frames 14 to 63
Iteration 4: At live edge - wait 5.0s before calling again
Waiting 5.0s at live edge...
Checking frames 14 to 63
✓ Viewer updated - showing 14 frames
Iteration 5: Loaded 4 frames, hit live edge
Checking frames 18 to 67
✓ Viewer updated - showing 18 frames
Iteration 6: Loaded 2 frames, hit live edge
Checking frames 20 to 69
Iteration 7: At live edge - wait 5.0s before calling again
Waiting 5.0s at live edge...
Checking frames 20 to 69
✓ Viewer updated - showing 20 frames
Iteration 8: Loaded 4 frames, hit live edge
Checking frames 24 to 73
✓ Viewer update

KeyboardInterrupt: 